# AI Agents – From LLMs to Tool Use and Simple Reasoning

## Scenario: A Travel Assistant for Venice

In this notebook, we build a simple AI-based travel assistant step by step.

We start with a local Large Language Model (LLM) and investigate what it can and cannot answer on its own. In particular, we consider questions that require **current or external information**, such as:

* What will the weather be like in Venice next Saturday?
* Is a museum open at a particular time?
* Which additional information is relevant for answering a user's question?

We then extend the system with external information sources and simple decision-making capabilities.

> **Important:** The goal of this notebook is not to build a sophisticated autonomous agent. Instead, we use a deliberately simple example to understand the transition from an **LLM**, to an **AI workflow**, and finally to a **simple AI agent that selects between available tools**.

## Learning Objectives

After completing this notebook, you should be able to:

1. **Explain the limitations of an LLM without access to external information**, especially for current or dynamic information.

2. **Use external tools and APIs** to obtain information that is not available to the LLM itself.

3. **Explain the difference between an LLM and an AI workflow**, in which external information is collected according to a predefined sequence of steps.

4. **Explain the basic idea of tool use in AI agents**, where an LLM selects which external information or service is relevant for a given task.

5. **Distinguish between predefined workflows and simple agentic decision-making.**

6. **Recognize limitations of simple tool-using agents**, for example restricted tool choices, unreliable model outputs, missing information, and dependence on external services.

7. **Reflect on the role of reasoning and decision-making in AI agents:**
   When does an application merely execute a predefined workflow, and when does the AI model itself influence which action is performed?

## Roadmap

During the notebook, we gradually move through three stages:

**1. LLM only**
The model answers based on its internal knowledge, but has no access to current weather, opening hours, or other external resources.

**2. AI workflow**
External information is collected using predefined program logic and then provided to the LLM.

**3. Simple (iterative) AI agent**
The LLM is given a choice between different information sources and decides which one is most useful for answering the user's question.

The central question throughout the notebook is:

> **What needs to be added to an LLM before we can reasonably describe the resulting system as an AI agent?**




## 1. LLM only

We first need to do a couple of steps to prepare the execution of the following code, most importantly we need to

* install `Ollama` which locally installs and runs different open source LLMs, after installation you can check if the service runs: Type
> "ollama --version"

  in bash
* get the large language model `phi3` (Microsoft), type
  > "ollama pull phi3"

  in bash
* launch the model from bash, type
  > "ollama run phi3"
* pip install `langchain` and `langchain-ollama` (see below)
* install the `OSM opening-hours` parser used later in the notebook:
  > "pip install opening-hours-py"

We then invoke the LLM

In [9]:
#!pip install -U langchain langchain-ollama opening-hours-py
from langchain_ollama import OllamaLLM  # Note: do not import via langchain_community (deprecated)

llm = OllamaLLM(model="phi3") # qwen3:0.6b

We can use this model similar to ChatGPT, e.g.

In [11]:
text = llm.invoke("Who won the most Olympic gold medals in history")
print(text.split("\n")[0])

Michael Phelps won the most Olympic gold medals in history.


OK, what if we do the following?

In [13]:
text = llm.invoke("I want to go swimming in the sea near Venice next Saturday. Given the weather, is this a good idea?")
print(text.split("\n")[0])

As of my last update in April 2023, I can't provide real-time data. However, to determine whether swimming in the sea near Venice next Saturday is a good idea, you'll need to check the current and forecasted weather conditions for that specific day. Here'seldfollow these steps:


...so, the LLM cannot help us here, since it does not have access to the relevant ressources! We need some help!

In [14]:
#!pip install duckduckgo-search 
#!pip install ddgs
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
print(search.run("I want to go swimming in the sea near Venice next Saturday. Given the weather, is this a good idea?"))

Today, waves in Venice reach a maximum height of 1' 10" (0.55 m), and sea conditions are suitable for swimming . The sea is expected to remain calm over the next few days. Venice Sea Conditions table showing wave height, swell direction and period. High and low tide times are also provided on the table along with the moon phase and forecast weather . Oct 10, 2020 · Discover the best moment to go to the beach in Venice in Italy this week to enjoy the best weather and calmest sea possible. Our opinion is based on reliable sea weather forecasts in Venice . Live weather reports from Venice weather stations and weather warnings that include risk of thunder, high UV index and forecast gales. See the links below the 12-day Venice weather forecast table for other cities and towns nearby along with weather conditions for local outdoor activities. Sep 23, 2026 · Do you want to swim in the sea in Venice in Italy today? First, check if the sea temperature is appropriate for swimming with our data 

## 2. AI Workflow

This simple search is not too helpful, let's see if we can use a weather API and system tools to collect more info. Let us assume that the parameters for which we want to make a query are the place and the date.

We first get the GPS coordinates of the place (in this case: "Venice") and then the date for "next Saturday" before querying the weather API:

In [15]:
#!pip install geopy

from geopy.geocoders import Nominatim

def get_coordinates(place: str):
    geolocator = Nominatim(user_agent="geoapi_ex")
    location = geolocator.geocode(place)
    if location:
        return location.latitude, location.longitude
    else:
        return None

# example:
coords = get_coordinates("Venice, Italy")
if coords:
    print(f"Venice, Italy coordinates: {coords[0]:.4f}, {coords[1]:.4f}")
else:
    print("Location not found")

Venice, Italy coordinates: 45.4372, 12.3346


We then need to find out which date "next Saturday" refers to.

In [16]:
from datetime import datetime, timedelta
import calendar

def get_date_of_today():
    return datetime.today()

def get_offset(day: str, iso: bool = False) -> int:
    days = list(calendar.day_name)  # ["Monday", "Tuesday", ...]
    idx = days.index(day)
    return idx + 1 if iso else idx

def get_date_for_next_weekday(weekday, from_date=None):
    if from_date is None:
        from_date = get_date_of_today()
    offset = get_offset(weekday) # get the id of the day we are interested in: Mon=0 ... Sun=6
    days_ahead = offset - from_date.weekday()  # compute the delta in days: weekday(): Mon=0 ... Sun=6, so Sat=5
    if days_ahead <= 0:  # if today is Saturday or later
        days_ahead += 7
    return (from_date + timedelta(days=days_ahead)).date()

# example:
print("Next Saturday:", get_date_for_next_weekday("Saturday"))

Next Saturday: 2026-10-10


We then query a weather API (here: `Open Meteo`) with said geo location and date:

In [17]:
import requests

def get_weather_forecast(lat, lon, date):
    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={lat}&longitude={lon}"
        "&hourly=temperature_2m,weathercode"
        "&timezone=auto"
    )
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()

    forecasts = []
    for t, temp, code in zip(data["hourly"]["time"], 
                             data["hourly"]["temperature_2m"], 
                             data["hourly"]["weathercode"]):
        if t.startswith(date):
            forecasts.append({
                "time": t[-5:],  # Uhrzeit HH:MM
                "temp": temp,
                "code": code
            })
    return forecasts

# example for Venice and next Saturday:
forecasts = get_weather_forecast(coords[0], coords[1], f"{get_date_for_next_weekday("Saturday")}")
for f in forecasts:
    print(f"{f['time']}: {f['temp']}°C, Wettercode {f['code']}")


00:00: 16.8°C, Wettercode 2
01:00: 16.5°C, Wettercode 2
02:00: 16.1°C, Wettercode 2
03:00: 15.8°C, Wettercode 2
04:00: 15.6°C, Wettercode 2
05:00: 15.5°C, Wettercode 2
06:00: 15.2°C, Wettercode 2
07:00: 15.3°C, Wettercode 3
08:00: 15.4°C, Wettercode 3
09:00: 15.9°C, Wettercode 2
10:00: 17.0°C, Wettercode 3
11:00: 18.5°C, Wettercode 2
12:00: 20.0°C, Wettercode 1
13:00: 21.4°C, Wettercode 3
14:00: 22.3°C, Wettercode 3
15:00: 21.8°C, Wettercode 3
16:00: 21.6°C, Wettercode 3
17:00: 21.5°C, Wettercode 3
18:00: 21.0°C, Wettercode 3
19:00: 20.0°C, Wettercode 3
20:00: 19.5°C, Wettercode 3
21:00: 19.3°C, Wettercode 3
22:00: 18.9°C, Wettercode 3
23:00: 18.8°C, Wettercode 3


Ok, so now we have the temperature in Venice for next Saturday, an info our AI workflow can obtain but our LLM alone cannot.
We now feed this back to the LLM and request a judgement:

In [18]:
# just use temperature at noon
temp = forecasts[12]['temp']
print(f"Temperature at noon is: {temp}")

# and feed this info back to the LLM
text = llm.invoke(f"I want to go swimming in the sea near Venice next Saturday. Given the weather, is this a good idea? The air temperature is predicted to be {temp}. Make a judgement based on this info alone. ")
print(text.split("\n")[0])

Temperature at noon is: 20.0
Given that the air temperature is predicted to be 2degrees Celsius (20.0 in Fahrenheit), it suggests mild weather conditions. However, swimming in the sea, particularly near Venice, also depends on other factors such as water temperature, wave conditions, and safety regulations like tides and currents. Mild air temperatures could mean the sea might be cool, which might not be comfortable for swimming. If the water is not sufficiently warm, it's generally not advisable to swim. Additionally, you should check local weather forecasts for sea conditions and any advisories or warnings. As a rule of thumb, ensure that the sea conditions are calm and that you adhere to safety guidelines.


So - now we get a more informed recommendation. This is an *AI workflow*: the program decides in advance which external information to retrieve, and the LLM uses that information to formulate a response.

Of course, the question is: **Did the additional information actually suffice to answer the question?**

Probably not. For example, we have so far ignored water temperature, wind, waves, and possible weather warnings.

> **Reflection:** Which parts of this workflow are fixed by the programmer, and which part is performed by the LLM?


We now implement a second external service: opening hours for POIs in Venice.

> **Implementation note:** Nominatim/OSM returns opening hours in the expressive OSM `opening_hours` syntax. Instead of writing our own parser, we use the dedicated `opening-hours-py` package to evaluate this syntax for the **actual date** we are interested in. This also handles rules and exceptions such as `We-Mo 10:00-18:00, Tu,Dec 25 off` correctly.


In [21]:
#!pip install opening-hours-py

import requests
from datetime import datetime, timedelta, time
from zoneinfo import ZoneInfo

from opening_hours import OpeningHours, State

# ---------- POI lookup: Nominatim + OSM opening_hours ----------
NOMINATIM = "https://nominatim.openstreetmap.org/search"
OVERPASS = "https://overpass-api.de/api/interpreter"

HEADERS = {"User-Agent": "ai-agents-teaching-notebook/1.0"}


def geocode_poi_in_venice(poi_name: str):
    """
    Find the POI in Venice.

    extratags=1 is important here: Nominatim may then return the OSM
    opening_hours value directly.
    """
    q = f"{poi_name}, Venice, Italy"

    response = requests.get(
        NOMINATIM,
        params={
            "q": q,
            "format": "jsonv2",
            "limit": 1,
            "extratags": 1,
        },
        headers=HEADERS,
        timeout=20,
    )
    response.raise_for_status()

    hits = response.json()
    return hits[0] if hits else None


def overpass_get_opening_hours(osm_type: str, osm_id: int, timeout=25):
    """
    Fallback: retrieve opening_hours from the exact OSM object via Overpass.
    """
    osm_type = osm_type.lower()
    element_type = {
        "node": "node",
        "way": "way",
        "relation": "relation",
    }[osm_type]

    query = f"""
    [out:json][timeout:{timeout}];
    {element_type}({osm_id});
    out tags;
    """

    response = requests.get(
        OVERPASS,
        params={"data": query},
        headers=HEADERS,
        timeout=timeout,
    )
    response.raise_for_status()

    elements = response.json().get("elements", [])
    if not elements:
        return None

    return elements[0].get("tags", {}).get("opening_hours")


def normalize_opening_hours(raw: str) -> str:
    """
    Correct a few known malformed OSM opening_hours expressions.
    Do not globally replace commas, because commas can be valid
    parts of OSM opening_hours syntax.
    """
    raw = raw.strip()

    known_fixes = {
        "We-Mo 10:00-18:00, Tu off,Dec 25 off":
            "We-Mo 10:00-18:00; Tu off; Dec 25 off",

        "We-Mo 10:00-18:00, Tu off, Dec 25 off":
            "We-Mo 10:00-18:00; Tu off; Dec 25 off",

        "We-Mo 10:00-18:00, Tu,Dec 25 off":
            "We-Mo 10:00-18:00; Tu off; Dec 25 off",
    }

    return known_fixes.get(raw, raw)
    

def evaluate_opening_hours_for_date(
    opening_hours_raw: str,
    target_date,
    lat: float,
    lon: float,
):
    timezone = ZoneInfo("Europe/Rome")

    opening_hours_raw = normalize_opening_hours(opening_hours_raw)

    try:
        schedule = OpeningHours(
            opening_hours_raw,
            timezone=timezone,
            coords=(float(lat), float(lon)),
        )
    except Exception as e:
        print("Could not parse opening_hours:")
        print(" ", repr(opening_hours_raw))
        print(" ", e)
        return []

    day_start = datetime.combine(
        target_date,
        time.min,
        tzinfo=timezone,
    )
    day_end = day_start + timedelta(days=1)

    slots = []

    for interval_start, interval_end, state, comment in schedule.intervals(
        start=day_start,
        end=day_end,
    ):
        if state != State.OPEN:
            continue

        start = max(interval_start or day_start, day_start)
        end = min(interval_end or day_end, day_end)

        if start < end:
            slots.append({
                "opening_hours_start": start.strftime("%H:%M"),
                "opening_hours_end": end.strftime("%H:%M"),
            })

    return slots


def opening_hours_start_end_for_poi_day(poi_name: str, weekday_name: str):
    """
    Return opening intervals for the NEXT occurrence of weekday_name.

    The raw OSM opening_hours expression is evaluated for the concrete date,
    so date-specific exceptions such as 'Dec 25 off' are respected.
    """
    geo = geocode_poi_in_venice(poi_name)

    if not geo:
        return {
            "poi": poi_name,
            "weekday": weekday_name,
            "date": None,
            "opening_hours_raw": None,
            "slots": [],
            "note": "POI not found by Nominatim.",
        }

    # First try the opening_hours value already returned by Nominatim.
    opening_hours_raw = (
        geo.get("extratags") or {}
    ).get("opening_hours")

    source = "Nominatim extratags"

    # Fallback: query the exact OSM object.
    if not opening_hours_raw:
        opening_hours_raw = overpass_get_opening_hours(
            geo["osm_type"],
            int(geo["osm_id"]),
        )
        source = "Overpass"

    target_date = get_date_for_next_weekday(weekday_name)

    if not opening_hours_raw:
        return {
            "poi": geo.get("display_name", poi_name),
            "weekday": weekday_name,
            "date": str(target_date),
            "opening_hours_raw": None,
            "slots": [],
            "osm_ref": f"{geo['osm_type']}/{geo['osm_id']}",
            "source": source,
            "note": "No opening_hours value was found in OSM.",
        }

    slots = evaluate_opening_hours_for_date(
        opening_hours_raw,
        target_date,
        float(geo["lat"]),
        float(geo["lon"]),
    )

    return {
        "poi": geo.get("display_name", poi_name),
        "weekday": weekday_name,
        "date": str(target_date),
        "opening_hours_raw": opening_hours_raw,
        "slots": slots,
        "osm_ref": f"{geo['osm_type']}/{geo['osm_id']}",
        "source": source,
        "lat": geo.get("lat"),
        "lon": geo.get("lon"),
    }


# Example / diagnostic output
for poi in [
    "Peggy Guggenheim Collection",
    "Gallerie dell'Accademia",
]:
    info = opening_hours_start_end_for_poi_day(
        poi,
        "Saturday",
    )

    print("\n" + poi)
    print("Date:", info.get("date"))
    print("OSM reference:", info.get("osm_ref"))
    print("Source:", info.get("source"))
    print("Raw opening_hours:", info.get("opening_hours_raw"))
    print("Parsed slots:", info.get("slots"))
    if info.get("note"):
        print("Note:", info["note"])



Peggy Guggenheim Collection
Date: 2026-10-10
OSM reference: node/5366368715
Source: Nominatim extratags
Raw opening_hours: We-Mo 10:00-18:00, Tu off,Dec 25 off
Parsed slots: [{'opening_hours_start': '10:00', 'opening_hours_end': '18:00'}]

Gallerie dell'Accademia
Date: 2026-10-10
OSM reference: node/895805118
Source: Nominatim extratags
Raw opening_hours: Tu-Su 08:15-19:15; Mo 08:15-14:00
Parsed slots: [{'opening_hours_start': '08:15', 'opening_hours_end': '19:15'}]


Looks something like this:

> Peggy Guggenheim Collection
Date: 2026-08-15
OSM reference: node/5366368715
Source: Nominatim extratags
Raw opening_hours: We-Mo 10:00-18:00, Tu,Dec 25 off

Okay, so now we have two services available:

- weather information
- opening hours

In the previous workflow, the program decided which information to retrieve. We now let the **LLM choose which of these two tools is more relevant for the user's question**.

This is still a deliberately simple setup: the available tools and most parameters are predefined by the programmer.


In [22]:
query = "I want to go the Peggy Guggenheim Collection in Venice next Saturday at 16:00. what do you think?"
text = llm.invoke(query)
print(text.split("\n")[0])

The Peggy Guggenheim Collection in Venice is one of the world's most important museums for modern art, particularly known for its impressive assemblage of 20th-century art. Since you're aiming to visit at 16:0CT, here are some points to consider:


...the LLM just returns some basic info about the museum. We want to improve a bit:

In [23]:
# We first ask the LLM which additional information would be more useful:
# 0 = weather, 1 = museum opening hours.
query = "I want to go the Peggy Guggenheim Collection in Venice next Saturday at 16:00. What do you think?"

routing_prompt = (
    "The query is: " + query +
    ". Decide which additional information is most useful to answer the query. "
    "Return either '0' or '1' and no additional text. "
    "0 means: get air temperature in Venice that day. "
    "1 means: get opening hours of the museum."
)

selection = llm.invoke(routing_prompt).strip().split("\n")[0].strip()
print("Selection for one additional query is " + selection)

if selection not in {"0", "1"}:
    raise ValueError(f"Unexpected tool selection: {selection!r}")

if selection == "0":
    print("0: requesting weather forecast, not opening hours...")

    # The query refers to Saturday, so obtain the Saturday forecast.
    saturday = get_date_for_next_weekday("Saturday")
    saturday_forecasts = get_weather_forecast(coords[0], coords[1], f"{saturday}")
    temp = saturday_forecasts[12]["temp"]
    additional_info = f" Air temperature in Venice on that day is {temp}°C."

else:
    print("1: requesting opening hours, not weather forecast...")

    # The query refers to Saturday.
    opening_hours_info = opening_hours_start_end_for_poi_day(
        "Peggy Guggenheim Collection", "Saturday"
    )
    if opening_hours_info["slots"]:
        start = opening_hours_info["slots"][0]["opening_hours_start"]
        end = opening_hours_info["slots"][0]["opening_hours_end"]
        print("Start: " + start + ", end: " + end)
        additional_info = (
            " On that day the opening hours are from " + start + " to " + end + "."
        )
    else:
        print("No parseable opening hours returned.")
        print("Raw opening_hours:", opening_hours_info.get("opening_hours_raw"))
        additional_info = (
            " No reliable opening-hours interval could be retrieved for that day."
        )

# Query the LLM again, now with the selected external information.
text = llm.invoke(query + additional_info)
print(text.split("\n")[0])


Selection for one additional query is 1
1: requesting opening hours, not weather forecast...
Start: 10:00, end: 18:00



We have now created a **minimal tool-routing agent**.

The LLM does not merely formulate the final answer; it also makes one simple decision about **which available tool to use**. However, the setup is still highly constrained:

- only two tools are available,
- the tool descriptions are explicitly provided,
- tool parameters such as the museum name and weekday are still hard-coded in our program.

> **Reflection:** What has changed compared with the workflow above? What is still predefined by the programmer?


In [24]:
# Try the same tool-routing idea for the earlier swimming question.
query = "I want to go swimming in the sea near Venice next Saturday. What do you think?"

routing_prompt = (
    "The query is: " + query +
    ". Decide which additional information is most useful to answer the query. "
    "Return either '0' or '1' and no additional text. "
    "0 means: get air temperature in Venice that day. "
    "1 means: get opening hours of the museum."
)

selection = llm.invoke(routing_prompt).strip().split("\n")[0].strip()
print("Selection is " + selection)

if selection not in {"0", "1"}:
    raise ValueError(f"Unexpected tool selection: {selection!r}")

if selection == "0":
    print("0: requesting weather forecast, not opening hours...")
    temp = forecasts[12]["temp"]
    additional_info = f" Air temperature in Venice on that day is {temp}°C."

else:
    print("1: requesting opening hours, not weather forecast...")
    opening_hours_info = opening_hours_start_end_for_poi_day(
        "Peggy Guggenheim Collection", "Saturday"
    )
    if opening_hours_info["slots"]:
        start = opening_hours_info["slots"][0]["opening_hours_start"]
        end = opening_hours_info["slots"][0]["opening_hours_end"]
        print("Start: " + start + ", end: " + end)
        additional_info = (
            " On that day the opening hours are from " + start + " to " + end + "."
        )
    else:
        print("No parseable opening hours returned.")
        print("Raw opening_hours:", opening_hours_info.get("opening_hours_raw"))
        additional_info = (
            " No reliable opening-hours interval could be retrieved for that day."
        )

# Query the LLM again with the selected information.
text = llm.invoke(query + additional_info)
print(text.split("\n")[0])


Selection is 0
0: requesting weather forecast, not opening hours...
As an AI, I don't have personal feelings, but I can provide you with information to help you make an informed decision. If the air temperature in Venice on a Saturday is 2curiously, I notice that you've asked about Venice with a temperature of 20.0°C, which is quite mild and pleasant. Typically, sea temperatures in the Venetian Lagoon are influenced by the weather and can vary. On a day with mild air temperatures, it's reasonable to expect that the sea temperature might also be comfortable for swimming. However, it's important to check the specific sea temperature for next Saturday, as it may differ from the air temperature due to various factors like recent weather patterns, wind, and water currents.


## 4. Reflection: From Workflow to Tool Routing

Discuss briefly:

1. Why is the first weather example an **AI workflow** rather than an agent?
2. What additional decision is made by the LLM in the tool-routing example?
3. What is still hard-coded and therefore *not* decided by the agent?
4. What could go wrong if the LLM returns something other than exactly `0` or `1`?
5. Does selecting the right tool guarantee a correct final answer? Why not?



### Solution

**Why is the first weather example an AI workflow rather than an agent?**
The sequence of actions is completely predefined by the programmer: determine the location, determine the date, call the weather API, select the temperature, and provide it to the LLM. The LLM does not decide which external information to obtain or which tool to use. It only generates the final response based on the information supplied to it.

**What additional decision is made by the LLM in the tool-routing example?**
The LLM decides which of the available external information sources is more useful for answering the user's question. In the example, it chooses between retrieving weather information and retrieving museum opening hours. Thus, part of the control flow is now determined by the model rather than entirely by predefined program logic.

**What is still hard-coded and therefore not decided by the agent?**
Several aspects are still predefined, for example:

* only two tools are available,
* the meaning of `0` and `1` is fixed,
* the code specifies how each tool is called,
* the location and POI are partly hard-coded,
* the program determines what happens after a tool has been selected,
* only one additional source of information is retrieved.

The agent therefore has only a very limited degree of autonomy.

**What could go wrong if the LLM returns something other than exactly `0` or `1`?**
The program expects a very specific output format. If the LLM returns something such as `"1 - opening hours"`, `"I choose 1"`, or an unexpected value, converting the output with `int(selection)` may fail or the program may follow the wrong branch. This illustrates why relying on free-text model output for program control can be fragile. More robust agent systems often use structured outputs or explicit tool-calling mechanisms instead.

**Does selecting the right tool guarantee a correct final answer? Why not?**
No. Selecting the appropriate tool is only one step. The tool may return incomplete, outdated, or incorrect information, the program may extract the wrong part of the result, or the LLM may interpret the information incorrectly. The LLM may also add unsupported claims to its final response.

For example, retrieving the air temperature is relevant to the question of whether swimming is a good idea, but air temperature alone may not be sufficient. Water temperature, wind, waves, weather warnings, and other conditions could also matter.

### Key idea

The transition from a workflow to an agent is gradual rather than absolute. In this example, the system becomes more agent-like because the LLM influences **which action is performed next**, but much of the possible behavior is still constrained by hard-coded program logic.


## 5. From Tool Routing to a Simple Iterative Agent

So far, the LLM selects **one** additional source of information and the program then immediately produces the final answer.

A slightly more agent-like system can repeat this process:

> **decide → use a tool → observe the result → decide again**

The LLM may choose between three actions:

- `weather` – retrieve weather information,
- `opening_hours` – retrieve museum opening hours,
- `finish` – stop collecting information and answer the user.

The available tools are still predefined by the programmer. However, the LLM now decides **which tool to use next and when it has collected enough information**.

To keep the example simple and transparent, we limit the number of steps and use the same Venice/Saturday scenario as above.


In [25]:
import re

# ============================================================
# 1. Tools available to the agent
# ============================================================

def weather_tool():
    """
    Return a compact weather observation for Venice next Saturday.
    """
    
    saturday = get_date_for_next_weekday("Saturday")

    forecasts = get_weather_forecast(
        coords[0],
        coords[1],
        str(saturday)
    )

    # For this simple example, use the air temperature around noon.
    if len(forecasts) <= 12:
        return "No reliable weather information could be retrieved."

    temp = forecasts[12]["temp"]

    return (
        f"Air temperature in Venice around noon "
        f"on Saturday: {temp} °C."
    )


def opening_hours_tool():
    """
    Return opening hours for the Peggy Guggenheim Collection
    next Saturday.
    """

    info = opening_hours_start_end_for_poi_day(
        "Peggy Guggenheim Collection",
        "Saturday"
    )

    # The external service may return no usable opening interval.
    if not info["slots"]:
        raw = info.get("opening_hours_raw")

        if raw:
            return (
                "Opening-hours information was found, "
                f"but no opening interval could be determined. "
                f"Raw OSM value: {raw}"
            )

        return (
            "No reliable opening-hours information "
            "could be retrieved for Saturday."
        )

    start = info["slots"][0]["opening_hours_start"]
    end = info["slots"][0]["opening_hours_end"]

    return (
        "Peggy Guggenheim Collection opening hours "
        f"on Saturday: {start}–{end}."
    )


# ============================================================
# 2. Robustly extract the action selected by the LLM
# ============================================================

def extract_action(llm_output, used_tools=None):
    """
    Extract one of the allowed actions from an LLM response.

    The LLM is instructed to return only one word, but smaller
    language models may nevertheless return a full explanation.

    This function therefore tries to recover the intended action.
    """

    if used_tools is None:
        used_tools = set()

    text = llm_output.strip().lower()

    valid_actions = {
        "weather",
        "opening_hours",
        "finish"
    }

    # --------------------------------------------------------
    # Case 1:
    # Ideal situation: the LLM returned exactly one action.
    # --------------------------------------------------------
    if text in valid_actions:
        return text

    # --------------------------------------------------------
    # Case 2:
    # The LLM wrote something such as:
    # "Action: weather"
    # --------------------------------------------------------
    match = re.search(
        r"(?:action|choice|select|selection)\s*[:=-]?\s*"
        r"(weather|opening_hours|finish)",
        text
    )

    if match:
        action = match.group(1)

        # Do not repeat a tool that has already been used.
        if action not in used_tools:
            return action

    # --------------------------------------------------------
    # Case 3:
    # The LLM produced a longer explanation.
    #
    # Example:
    # "We already used opening_hours. We should now use weather
    # and afterwards finish."
    #
    # Find all action words in the order in which they occur.
    # Ignore tools that have already been used.
    # --------------------------------------------------------
    mentioned_actions = re.findall(
        r"\b(opening_hours|weather|finish)\b",
        text
    )

    for action in mentioned_actions:

        if action == "finish":
            return "finish"

        if action not in used_tools:
            return action

    # No valid decision could be extracted.
    return None




In [26]:
# ============================================================
# 3. Simple iterative agent
# ============================================================

def run_iterative_agent(query, max_steps=3):
    """
    A deliberately simple iterative agent.

    At every step the LLM decides whether to:

        weather
        opening_hours
        finish

    The selected tool is executed and its result is added to
    the observations. The LLM then decides again.

    The process stops when:
    - the LLM selects 'finish', or
    - max_steps is reached.
    """

    observations = []
    used_tools = set()

    # --------------------------------------------------------
    # Agent loop
    # --------------------------------------------------------

    for step in range(max_steps):

        if observations:
            known_information = "\n".join(
                f"- {obs}" for obs in observations
            )
        else:
            known_information = (
                "No external information has been collected yet."
            )

        available_tools = []

        if "opening_hours" not in used_tools:
            available_tools.append("opening_hours")

        if "weather" not in used_tools:
            available_tools.append("weather")

        available_tools.append("finish")

        routing_prompt = f"""
The user asks:

{query}

Information collected so far:

{known_information}

Available actions:

{", ".join(available_tools)}

Choose the ONE most useful NEXT action.

Rules:
- Do not choose a tool that has already been used.
- Use opening_hours if museum opening hours are needed.
- Use weather if weather information is needed.
- Choose finish only when enough relevant information has been collected.
- Return only ONE action name.

Valid examples:

weather

opening_hours

finish
"""

        # Ask the LLM what to do next.
        raw_action = llm.invoke(routing_prompt)

        # Try to obtain a valid program action from the LLM response.
        action = extract_action(
            raw_action,
            used_tools=used_tools
        )

        print(f"\nStep {step + 1}")
        print("LLM response:", raw_action.strip())
        print("Selected action:", action)

        # ----------------------------------------------------
        # Could not interpret the LLM output
        # ----------------------------------------------------

        if action is None:
            print(
                "No valid action could be extracted. "
                "Stopping the agent."
            )
            break

        # ----------------------------------------------------
        # Agent decides that it has enough information
        # ----------------------------------------------------

        if action == "finish":
            print("Agent decides that enough information is available.")
            break

        # ----------------------------------------------------
        # Execute selected tool
        # ----------------------------------------------------

        if action == "weather":

            observation = weather_tool()

        elif action == "opening_hours":

            observation = opening_hours_tool()

        else:
            # Should normally never be reached.
            print("Unknown action. Stopping.")
            break

        # Remember that this tool has already been used.
        used_tools.add(action)

        # Add tool result to the agent's observations.
        observations.append(observation)

        print("Observation:", observation)

    # ========================================================
    # 4. Generate final answer
    # ========================================================

    if observations:
        evidence = "\n".join(
            f"- {obs}" for obs in observations
        )
    else:
        evidence = "No external information was collected."

    answer_prompt = f"""
The user asks:

{query}

The agent collected the following external information:

{evidence}

Answer the user's question based on the supplied information.

Important:
- Use the external information above.
- Do not invent additional current facts.
- If the available information is insufficient for part of the
  question, explicitly say so.
"""

    answer = llm.invoke(answer_prompt)

    return answer, observations

### Test: a query that may require more than one tool

Consider a question that combines two activities:

> *“I want to visit the Peggy Guggenheim Collection in Venice next Saturday afternoon and go swimming afterwards. Is that a good plan?”*

Unlike the previous examples, a useful answer may require **more than one type of external information**.

Run the agent and observe:

1. Which tool does it select first?
2. Does it request the second tool afterwards?
3. When does it decide to `finish`?
4. Does the final answer stay within the information actually collected?


In [30]:
query = (
    "I want to visit the Peggy Guggenheim Collection in Venice "
    "next Saturday afternoon and go swimming afterwards. "
    "Is that a good plan?"
)

answer, observations = run_iterative_agent(
    query,
    max_steps=3
)


print("\n\nCollected observations:")

for observation in observations:
    print("-", observation)


print("\nFinal answer:")
print(answer)


Step 1
LLM response: opening_hours, because we need to confirm if the Peggy Guggenheim Collection is open next Saturday afternoon before planning further activities such as swimming afterwards. We can only choose finish once sufficient relevant information has been gathered through these actions or others that provide necessary data for a well-informed decision.
Selected action: opening_hours
Observation: Peggy Guggenheim Collection opening hours on Saturday: 10:00–18:00.

Step 2
LLM response: Weather, since we need to ensure the weather will be suitable for swimming after visiting the museum in the afternoon. Swimming outside could depend on factors like rain or excessively high temperatures which are not something one can predict without checking current conditions first. This information is essential before finalizing plans that include both indoor and outdoor activities, especially if they're back-to-back as requested by the user.
Selected action: weather
Observation: Air temperat

### Reflection

Discuss briefly:

1. What makes this system more agent-like than the previous tool-routing example?
2. Which decisions are now made repeatedly by the LLM?
3. What is still hard-coded by the programmer?
4. Why do we limit the number of steps? What could happen without such a limit?
5. Would adding more tools automatically make the agent better? Why or why not?



### Solution

1. **What makes this system more agent-like than the previous tool-routing example?**

    The system no longer makes only a single tool-selection decision. It repeatedly follows a loop of decide → act → observe → decide again. After each tool result, the LLM can determine whether more information is needed or whether it can stop and answer the user.

2. **Which decisions are now made repeatedly by the LLM?**
    At each step, the LLM decides:

    which available tool should be used next,
    whether another tool is still needed,
    or whether enough information has been collected and the agent should finish.

    The control flow therefore depends more strongly on the model's decisions.

3. **What is still hard-coded by the programmer?**

    Several important aspects are still predefined:

    * which tools exist (weather, opening_hours, finish),
    * what each tool does,
    * how the tools are implemented,
    * the location, museum, and weekday in the simplified tool wrappers,
    * the maximum number of steps,
    * the allowed action names,
    * how tool outputs are stored and passed back to the LLM,
    * how the final answer is generated.

    The agent therefore has only limited autonomy.

4. **Why do we limit the number of steps? What could happen without such a limit?**

    Without a limit, the agent could enter a loop, repeatedly request tools, or fail to decide that it has enough information. This could waste time and computational resources or cause unnecessary API calls. max_steps therefore acts as a simple safety mechanism.

5. **Would adding more tools automatically make the agent better? Why or why not?**

    No. More tools provide more possible capabilities, but they also make tool selection more difficult. The agent may choose the wrong tool, call unnecessary tools, misunderstand results, or become less reliable. Additional tools can also increase cost, latency, and implementation complexity.

    The important point is therefore not to provide as many tools as possible, but to provide relevant, reliable tools and a robust mechanism for choosing and using them.

### Key idea:

The system is more agent-like because the LLM now influences a sequence of actions rather than making only one isolated routing decision. However, the action space, available tools, stopping conditions, and execution logic are still largely controlled by the programmer.

*Citation*: In the generation of this notebook, ChatGPT5.1 was used, in particular to realize the API calls.